# iprPy viscosity_driving calculation

In [1]:
# Standard library imports
import datetime
from copy import deepcopy

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty, Image

import matplotlib.pyplot as plt

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-07-01 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('viscosity_driving')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# viscosity_driving calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The viscosity_driving calculation style estimates the viscosity of a liquid by applying a sinusoidal driving force to the material
### Version notes

- 2025-??: Initial version of the calculation added.
- 2026-: Calculation updated for the new LAMMPS interface.

### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- 

## Simulation design

The viscosity_driving calculation evaluates the viscosity of a liquid using the cosine periodic perturbation method. This uses the LAMMPS commands fix accelerate/cos and compute viscosity/cos to apply a periodic external acceleration to the atoms in the simulation, then estimate the viscosity based on the resulting velocity profile.  The simulation is sensitive to the amplitude of the acceleration used.

See the associated LAMMPS commands for more details: https://docs.lammps.org/fix_accelerate_cos.html and https://docs.lammps.org/compute_viscosity_cos.html.



## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "viscosity_driving.py"

# Python script created by Peter Winstel and Lucas Hale

# Standard Python libraries
from typing import Optional, Union

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj

def viscosity_driving(lammps_command: Union[str, LAMMPSobj],
                      system: am.System,
                      potential: lammpspotential,
                      temperature: float,
                      mpi_command: Optional[str] = None,
                      timestep: Optional[unitfloat] = None,
                      drivingforce: unitfloat = '2.0 angstrom/(ps^2)',
                      runsteps: int = 100000,
                      thermosteps: int = 100,
                      equilsteps: int = 0,
                      createvelocities: bool = False,
                      randomseed: Optional[int] = None,
                      usefiles: bool = False) -> dict:
    
    """
    Calculates the viscosity for a liquid system by applying a driving
    force.

    Parameters
    ----------
    lammps_command : str
        Command for running LAMMPS
    system : atomman.System
        The system to perform the calculation on.
    potential : atomman.lammps.Potential
        the LAMMPS implemented potential to use.
    temperature : float
        The temperature to run at.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel. If not given, LAMMPS
        will run serially.
    timestep : float, optional
        The amount of time to increase each frame of the simulation. The 
        default value is given by the default value for the specified LAMMPS
        unit system. 
    drivingforce : float, optional
        The amplitude of the driving force for the calculation method. Default 
        value is 2 angstrom/(ps^2). 
    runsteps : int, optional
        How many timesteps the simulation will run for. Default value of 100,000
        should be suitable for a short run. 
    thermosteps : int, optional
        How often the calculated values get stored in the thermo table of the 
        LAMMPS output. Default value of 1,000. 
    equilsteps : int, optional
        How many timesteps the equilibration simulation will run for. Default 
        value of 0.
    
    Returns
    -------
    dict
        Dictionary of results consisting of keys:

        -**'measured_temperature'** (*float*) - The average measured
        temperature of the system ignore initial data according to 
        the data offset.
        -**'measured_temperature_stderr'** (*float*) - The standard 
        deviation measured temperature of the system ignore initial 
        data according to the data offset.
        -**'viscosity'** (*float*) - The calculated viscosity 
        -**'viscosity_stderr'** (*float*) - The standard deviation
        of the viscosity
    """
    # Create a LAMMPS object if needed
    lmp = LAMMPS(lammps_command, mpi_command=mpi_command, potential=potential)

    logfile = 'log.lammps'
    if usefiles or not lmp.islib:
        script = 'viscosity_driving.in'
    else:
        script = None

    # Convert values given with units if needed
    drivingforce = uc.set_in_units(drivingforce)

    # Check/select a randomseed value
    randomseed = am.lammps.seed(randomseed)

    # Set timestep in atomman and LAMMPS units
    if timestep is None:
        timestep_lammps = am.lammps.style.timestep(lmp.potential.units)
        timestep = uc.set_in_units(timestep_lammps, lmp.unitsdict['time'])
    else:
        timestep = uc.set_in_units(timestep)
        timestep_lammps = uc.get_in_units(timestep, lmp.unitsdict['time'])
    temperature_damp = 100 * timestep_lammps


    # Pass system and potential info into LAMMPS
    lmp.new_system_from_data_file(system, filename='init.dat', tilt_large=True,
                                  usefiles=usefiles, logfile=logfile)

    lmp.commands_string('\n# I

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [12]:
lammps_command = 'F:/LAMMPS/current/bin/lmp.exe'
mpi_command = None
#mpi_command = 'mpiexec -localonly 6'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 23 Jun 2022 - Update 2


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial system

- __system__ is an atomman.System to use as the starting configuration.  Here, it is taken as the final configuration from the relax_liquid calculation.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Load final configuration from relax_liquid
system = am.load('atom_dump', '../relax_liquid/100000.dump', symbols='Ni')
print('# of atoms in system =', system.natoms)

# of atoms in system = 4000


### 3.4. Calculation-specific parameters

- __temperature__ is the temperature to run the calculation at.
- __timestep__ is the integration timestep to use. If None, then will use the default timestep for the LAMMPS units associated with the potential.
- __drivingforce__ is the amplitude of the driving force for the calculation method. Default value is 2 angstrom/(ps^2).
- __equilsteps__ is the number of integration steps to perform both before starting the diffusion calculations.  
- __runsteps__ is the number of integration steps to perform for the viscosity calculation.  Default value is 100000.
- __createvelocities__ indicates if new atomic velocities are to be assigned to the atoms prior to any MD runs.
- thermosteps__ indicates how often the calculated values get stored in the thermo table of the LAMMPS output. Default value of 1,000. 
- __randomseed__ is a random number seed between 1 and 9000000 to use for initializing velocities and use with the langevin thermostat.  Default value of None will pick a random value.

In [9]:
temperature = 2000.0
timestep = None
equilsteps = 0
runsteps = 100000
drivingforce = '2.0 angstrom/(ps^2)'
thermosteps = 1000
createvelocities = False
randomseed = None

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [10]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.viscosity_driving.viscosity_driving'

In [13]:
results_dict = calculation.calc(lammps_command, system, potential, temperature,
                                mpi_command = mpi_command,
                                timestep = timestep,
                                equilsteps = equilsteps,
                                runsteps = runsteps,
                                drivingforce = drivingforce,
                                thermosteps = thermosteps,
                                createvelocities = createvelocities,
                                randomseed = randomseed)
print(results_dict.keys())

dict_keys(['viscosity', 'viscosity_stderr', 'measured_temperature', 'measured_temperature_stderr'])


### 4.2. Report results

Values returned in the results_dict:

- **'viscosity'** (*float*) - The diffusion constant estimate obtained using the slope of the mean squared displacement averaged over all separate simulations.
- **'viscosity_stderr'** (*float*) - The diffusion constant estimate obtained from the mean squared displacement slope of the full combined simulation run.
- **'measured_temperature'** (*float*) - The mean observed temperature.
- **'measured_temperature_stderr'** (*float*) - The mean observed temperature.
- **'lammps_output'** (*atomman.lammps.Log*) - The LAMMPS thermo output, included to allow for further analysis.

In [17]:
viscosity_unit = 'Pa*ms'
print('μ  =', uc.get_in_units(results_dict['viscosity'], viscosity_unit), viscosity_unit)
print('μ_stderr =', uc.get_in_units(results_dict['viscosity_stderr'], viscosity_unit), viscosity_unit)

μ  = 7.09043894532073 Pa*ms
μ_stderr = 0.1293365281216189 Pa*ms


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [ ]:
calculation.clean_files()